# 200 simulated sales reps from a correlation matrix

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Random Number Generation** · a worked example from the StatNex teaching guide

**The story.** The correlation matrix of five sales-rep measures (meetings, emails, deals, revenue and rating) can also drive a simulation. This run uses it as a covariance matrix and draws 200 fake reps whose measures should be related in the same way. Deals closed and revenue correlate 0.95 in the matrix.

**The question.** Do simulated data reproduce the means, variances and correlations they were built from?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `12_utility/matrix_inverse.csv` (5 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Measure,Meetings per week,Emails per day,Deals closed,Revenue ($000),Customer rating
Meetings per week,1.0,0.3586,0.7429,0.7164,0.1357
Emails per day,0.3586,1.0,0.1832,0.1392,-0.2765
Deals closed,0.7429,0.1832,1.0,0.9462,0.149
Revenue ($000),0.7164,0.1392,0.9462,1.0,0.1354
Customer rating,0.1357,-0.2765,0.149,0.1354,1.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## Settings

In [ ]:
# count numbers in each of `columns` columns. The generator is numpy's default (PCG64),
# the same one the app uses, so with the same seed you get the app's numbers exactly
# (numpy may change a distribution's algorithm between releases, so that holds for
# the numpy version the app used). Without a seed (None) every run gives new numbers.
count = 200
columns = 10
seed = 20260809
rng = np.random.default_rng(seed)

## Draws

In [ ]:
# Multivariate normal with mean 0 and the covariance matrix in the data (a blank cell
# is 0; one typed triangle is mirrored). The matrix decides the number of columns.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
names = ["Meetings per week", "Emails per day", "Deals closed", "Revenue ($000)", "Customer rating"]
covariance = data[names].fillna(0).to_numpy(float)
upper = covariance[np.triu_indices(len(covariance), 1)]
lower = covariance[np.tril_indices(len(covariance), -1)]
if not upper.any() or not lower.any():
    covariance = covariance + covariance.T - np.diag(np.diag(covariance))
columns = len(covariance)
# Standard normals times the covariance's symmetric square root V·sqrt(L)·V', as the app
# draws them: rng.multivariate_normal uses an SVD instead, whose eigenvector signs depend
# on the computer's linear-algebra library, so the same seed could draw other numbers.
values, vectors = np.linalg.eigh(covariance)
root = (vectors * np.sqrt(np.clip(values, 0, None))) @ vectors.T
draws = rng.standard_normal((count, columns)) @ root
labels = names
theory = pd.DataFrame([np.zeros(columns), np.diag(covariance)],
                      index=["mean", "variance"], columns=labels)

## Theoretical mean and variance

In [ ]:
print(theory)

## Sample statistics of each column

In [ ]:
# The variance here is the unbiased one, dividing by n - 1 (ddof=1), as the app and the
# covariance matrix below do. Each column is one sample, so its statistics scatter
# around the theoretical values: about 95% of the column means should land within
# 2 * sqrt(variance / count) of the theoretical mean. Further out, suspect the
# settings, not the generator.
sample_statistics = pd.DataFrame([draws.mean(axis=0), draws.var(axis=0, ddof=1)],
                                 index=["mean", "unbiased variance"], columns=labels)
print()
print(sample_statistics)

## The numbers

In [ ]:
table = pd.DataFrame(draws, index=pd.RangeIndex(1, count + 1, name="No."), columns=labels)
table.to_csv("random_numbers.csv")
print()
print(table)

## Realised covariance matrix

In [ ]:
# np.cov divides by n - 1.
realised_covariance = np.cov(draws, rowvar=False)
print()
print(pd.DataFrame(realised_covariance, index=labels, columns=labels))

## What the result says

This run draws 200 fake reps from a multivariate normal distribution: five bell-shaped variables that are tied together by the given matrix. In theory each variable has a mean of 0 and a variance, a measure of spread, of 1.

The draws come close but do not match. The means run from -0.01 for emails to 0.10 for meetings, and the variances from 0.95 to 1.01. The realised variance-covariance matrix shows how the simulated variables move together. Deals and revenue have a covariance of 0.92, against a target of 0.95, and meetings and deals have 0.72 against 0.74.

These exact numbers come from the seed, the starting value of the random number generator, and another seed gives others. Simulated data hit their targets only on average, even with 200 draws.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.